# Milestone 3: Context Augmentation with RAG Pipelines

**Deadline:**  `Jul 10, 2026`

- The limitations of General LLMs. Learn the RAG pipeline in brief.
- Loading a simple pre-built vector database.
- Retrieving external context based on the question prompt.
- Feeding the retrieved context + prompt + choices into the model to improve reasoning.

<!--
gdown --folder 1QO1vigWXqH_FyHiOrXU94ogUlQivRzqj
 -->

## Setup

In [1]:
!uv pip install faiss-cpu #install FAISS

import faiss  # type: ignore
import numpy as np
import pandas as pd
from sentence_transformers import CrossEncoder, SentenceTransformer
from transformers import AutoTokenizer, pipeline

train = pd.read_csv("dataset/train.csv")

print("Creating nowledge base")
kb = []
for idx, row in train.iterrows():
    correct_letter = row["answer"]
    kb.append(str(row[correct_letter]))

print("Loading embedding model and creating index")
model = SentenceTransformer("all-MiniLM-L6-v2")
kb_embeddings = model.encode(kb, show_progress_bar=False)
index = faiss.IndexFlatL2(kb_embeddings.shape[1])
index.add(kb_embeddings)

print("Knowledge base successfully created")

Using Python 3.12.13 environment at: /usr
Checked 1 package in 93ms
Creating nowledge base
Loading embedding model and creating index


README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Knowledge base successfully created


### Zero-shot classifier for Q1, Q2, Q6

In [2]:
zs = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")
row_150 = train.iloc[150]
prompt_150 = str(row_150["prompt"])
labels_150 = [
    str(row_150["A"]),
    str(row_150["B"]),
    str(row_150["C"]),
    str(row_150["D"]),
    str(row_150["E"]),
]
ans_150 = str(row_150[row_150["answer"]])

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

## Question 1

Run the zero-shot classifier on `facebook/bart-large-mnli` on prompt for the row **index 150**. Pass the 5 options (A-E) candidate_labels. What is the predicted probability score assigned to the ground-truth correct option (option in the answer column)? (Round to 3 decimal points)

In [3]:
res = zs(prompt_150, candidate_labels=labels_150)

round(dict(zip(res["labels"], res["scores"]))[ans_150], 3)

0.384

## Question 2

Embed the prompt for row index 150 using `all-MiniLM-L6-v2`. Query your FAISS index to retrieve the top `k=10` most similar documents. At what exact rank (1 through 10) did FAISS place the true correct document (which is the document originally located at index 150 in the KB)?

In [4]:
query = model.encode([prompt_150])
D, I = index.search(query, k=10)

np.where(I[0] == 150)[0][0] + 1

np.int64(10)

## Concept: The Two-Stage Pipeline (Reranking & Cross-Encoders)

In Question 2, you saw that our FAISS database did not put the true document at rank #1. Why? Because FAISS uses Bi-encoder. \
A Bi-encoder embeds the question and the document separately and just compares the distance (Cosine Similarity). They are fast and allows you to search millions of documents but often miss semantic context.

To fix this we use a 2 stage pipeline:
1. **Retrieval:** Use a bi-encoder with FAISS to quickly get the top k possible chunks/documents.
2. **Reranking:** Use a Cross-Encoder to deeply evaluate those top candidates and sort them based on highest semantic similarity.

A Cross-Encoder passes the Question and the Document into the Transformer network at the exact same time. The Attention mechanism can directly compare the words in the question to the words in the document, resulting in a highly accurate relevance score.

For this we part the prompt and the context and ask it to predict a score.

[Cross Encoders (Hugging Face)](https://huggingface.co/cross-encoder)

### Code to use a Cross-Encoder

```python
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
docs_10 = [kb[i] for i in retrieved_indices] #Get the top 10 chunks
pairs = [[prompt_150, doc] for doc in docs_10] #Create prompt-context pairs
ce_scores = cross_encoder.predict(pairs) # Get the score of each pair
```

## Question 3

Take the top 10 documents retrieved by FAISS in the previous question. Load `cross-encoder/ms-marco-MiniLM-L-6-v2`. Score the prompt against these 10 documents and sort them by the cross-encoder's score. At what exact rank (1 through 10) does the Cross-Encoder place the true correct document?

In [5]:
ce = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
top_10 = [kb[i] for i in I[0]]
pairs = [[prompt_150, doc] for doc in top_10]

ce_score = ce.predict(pairs)  # type: ignore

sorted_ranks = enumerate(
    sorted(zip(I[0], ce_score), key=lambda x: x[1], reverse=True), start=1
)

list(filter(lambda x: x[1][0] == 150, sorted_ranks))[0][0]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

1

## Question 4

Retrieve the top `k=5` documents for the prompt at row index `42`. Concatenate them with a single space between each. Create a string: `"Context: [concatenated_docs] Question: [prompt]"`. Tokenize this string using the `bert-base-uncased` tokenizer (without truncation). Exactly how many total tokens does this generate?

In [6]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

row_42 = train.iloc[42]
prompt_42 = str(row_42["prompt"])
labels_42 = [
    str(row_42["A"]),
    str(row_42["B"]),
    str(row_42["C"]),
    str(row_42["D"]),
    str(row_42["E"]),
]
ans_42 = str(row_42[row_42["answer"]])

In [7]:
query = model.encode([prompt_42])
D, I = index.search(query, k=5)

top_5 = [kb[i] for i in I[0]]
context = " ".join(top_5)

text = f"Context: {context} Question: {prompt_42}"
len(tokenizer(text, truncation=False)["input_ids"])

216

## Question 5

Retrieve the exact true document for row **index 150** from your KB. Create a RAG string: `"Context: [true_document] Question: [prompt]"`. Run the same zero-shot classification from Question 1 on this augmented string. What is the new predicted probability score of the ground-truth correct option? (Round to 3 decimal places).

In [8]:
inp = f"Context: {ans_150} Question: {prompt_150}"

res = zs(inp, candidate_labels=labels_150)

round(dict(zip(res["labels"], res["scores"]))[ans_150], 3)

0.989

## Concept: The Danger of Bad Retrieval (Adversarial RAG)

The golden rule of Retrieval-Augmented Generation is **“Garbage In, Garbage Out.”** An LLM places immense trust in the external context you inject into its prompt. If your vector database performs poorly and retrieves an irrelevant or incorrect document, the model will often abandon its own internal reasoning and confidently generate the wrong answer based on that bad data. We do the reranking and constricting the number of chunks that we give to the model for the same reason.

## Question 6

What happens if your vector database retrieves the wrong information? Take the prompt for row **index 150**. Manually force the context to be the document located at KB **index 999** (a completely unrelated fact). Run the zero-shot classifier on this "Adversarial RAG" string. What is the probability of the correct option now? (Round to 3 decimal places).

In [9]:
inp = f"Context: {kb[999]} Question: {prompt_150}"

res = zs(inp, candidate_labels=labels_150)

round(dict(zip(res["labels"], res["scores"]))[ans_150], 3)

0.529

## Concept

In RAG, a "Hit" occurs if the retrieved context contains the facts needed to answer the question.

## Question 7

For the first 100 rows of train.csv (indices 0-99), retrieve the top `k=5` documents for each prompt. If the exact string of the row's correct option is found inside any of those 5 retrieved documents, it counts as a hit. What is the exact Hit Rate percentage (0 to 100) for these 100 rows? (Round to 1 decimal place).

In [10]:
hits = 0
for _, r in train.iloc[:100].iterrows():
    prompt = r["prompt"]
    ans = r[r["answer"]]

    q = model.encode([prompt])
    D, I = index.search(q, k=5)
    top_5 = [kb[i] for i in I[0]]
    if any(t == ans for t in top_5):
        hits += 1

round(float(hits), 1)

73.0

## Question 8

Build a loop that processes the first 20 rows (indices 0 through 19) of train.csv.
For each row, your pipeline must do the following in order:

- **Retrieve:** Embed the prompt and retrieve the top `k=5` documents from your FAISS Knowledge Base.

- **Rerank:** Pass the prompt and those 5 documents into the `ms-marco-MiniLM-L-6-v2` Cross-Encoder. Select the single document with the highest cross-encoder score.

- **Augment:** Create your RAG string exactly formatted as: "Context: [best_document] Question: [prompt]".

- **Predict:** Pass this augmented string to the `facebook/bart-large-mnli` zero-shot classifier, using the 5 options (A, B, C, D, E) as your `candidate_labels`.

- **Score:** Look at the probability scores output by the model. Rank the options from highest probability to lowest. Take the top 3 letters (e.g., `['C', 'A', 'E']`) and calculate the **MAP@3** for that row.

What is the final average **MAP@3** score of this state-of-the-art RAG pipeline across these 20 rows? (Round to 3 decimal places).

In [11]:
from typing import List, LiteralString


def map_3(truth: LiteralString, pred: List[LiteralString | str]) -> float:
    if truth not in pred:
        return 0.0

    return 1.0 / (pred.index(truth) + 1)

In [12]:
scores = []
for _, r in train.iloc[:20].iterrows():
    prompt = str(r["prompt"])

    labels = {
        "A": str(r["A"]),
        "B": str(r["B"]),
        "C": str(r["C"]),
        "D": str(r["D"]),
        "E": str(r["E"]),
    }

    ans = r["answer"]

    q = model.encode([prompt])
    D, I = index.search(q, k=5)
    top_5 = [kb[i] for i in I[0]]

    pairs = [[prompt, doc] for doc in top_5]
    ce_scores = ce.predict(pairs)  # type: ignore
    context = top_5[np.argmax(ce_scores)]

    rag_prompt = f"Context: {context} Question: {prompt}"
    res = zs(rag_prompt, candidate_labels=list(labels.values()))

    txt_to_ltr = {v: k for k, v in labels.items()}
    pred = [txt_to_ltr[l] for l in res["labels"]][:3]
    scores.append(map_3(ans, pred))

round(np.mean(scores), 3)

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


np.float64(0.975)